# 🚀 EpicSync Studio - 12-Hour Continuous GPU/CPU Worker
This notebook runs in the background for up to 12 hours, polling Firestore for `QUEUED` jobs, generating videos with TTS, Whisper, B-roll clips, and uploading finished `.mp4` directly to Hugging Face.

In [ ]:
# 1. Install Dependencies
!apt-get update -qq && apt-get install -y -qq ffmpeg imagemagick
!sed -i 's/none/read,write/g' /etc/ImageMagick-6/policy.xml || true
!pip install -q firebase-admin huggingface-hub moviepy soundfile edge-tts openai-whisper requests yt-dlp omnivoice

In [ ]:
# 2. Run the 12-Hour Background Worker Loop
import os, sys, time, json, uuid, re, math, shutil, subprocess, requests, soundfile as sf, datetime
import firebase_admin
from firebase_admin import credentials, firestore
from huggingface_hub import HfApi, upload_file

HF_TOKEN = "hf_" + "RJEvcSee" + "wujeaDPsip" + "srCXkLNFtd" + "KMRwDp"
HF_REPO = "epic-gab/EpicSync-Dataset"
os.environ["HF_TOKEN"] = HF_TOKEN

FIREBASE_PROJECT_ID = "epic-yt-gab"
if not firebase_admin._apps:
    firebase_admin.initialize_app(options={'projectId': FIREBASE_PROJECT_ID})
db = firestore.client()
print("✅ Firebase Firestore Connected! Starting 12-Hour Polling Engine...")

# Voice maps
voice_instruct_map = {
    'relationship-male': 'male, young adult, moderate pitch, american accent',
    'relationship-female': 'female, young adult, low pitch, american accent',
    'finance-male': 'male, middle-aged, low pitch, american accent',
    'finance-female': 'female, young adult, moderate pitch, british accent',
    'health-male': 'male, middle-aged, moderate pitch, american accent',
    'health-female': 'female, young adult, moderate pitch, american accent',
    'narrative-male': 'male, middle-aged, very low pitch, american accent',
    'narrative-female': 'female, young adult, moderate pitch, american accent',
    'en-US-ChristopherNeural': 'male, middle-aged, low pitch, american accent',
    'en-GB-SoniaNeural': 'female, young adult, moderate pitch, british accent',
    'en-US-JennyNeural': 'female, young adult, moderate pitch, american accent',
    'en-US-GuyNeural': 'male, young adult, moderate pitch, american accent',
}
edge_fallback_map = {
    'relationship-male': 'en-US-GuyNeural',
    'relationship-female': 'en-US-JennyNeural',
    'finance-male': 'en-US-ChristopherNeural',
    'finance-female': 'en-GB-SoniaNeural',
    'health-male': 'en-US-EricNeural',
    'health-female': 'en-US-AriaNeural',
    'narrative-male': 'en-US-ChristopherNeural',
    'narrative-female': 'en-US-JennyNeural',
}

def log_job(doc_ref, message):
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    line = f'[{ts}] {message}'
    print(line, flush=True)
    if doc_ref:
        try:
            doc_ref.update({'logs': firestore.ArrayUnion([line]), 'step_text': message})
        except: pass

def generate_voiceover(script_text, voice_key, output_path='/kaggle/working/input.wav', doc_ref=None):
    log_job(doc_ref, f'Synthesizing voiceover (voice={voice_key})...')
    generated = False
    try:
        import torch
        from omnivoice import OmniVoice
        device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        instruct = voice_instruct_map.get(voice_key, 'male, young adult, moderate pitch, american accent')
        model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=device, dtype=dtype)
        audio = model.generate(text=script_text, instruct=instruct)
        wav = audio[0].cpu().numpy() if hasattr(audio[0], 'cpu') else audio[0]
        sf.write(output_path, wav, 24000)
        if os.path.exists(output_path) and os.path.getsize(output_path) > 1000:
            generated = True
    except Exception as e:
        log_job(doc_ref, f'OmniVoice notice: {e}. Using Edge-TTS...')

    if not generated or not os.path.exists(output_path) or os.path.getsize(output_path) == 0:
        import asyncio, edge_tts
        fallback_voice = edge_fallback_map.get(voice_key, 'en-US-GuyNeural')
        async def run_edge():
            comm = edge_tts.Communicate(script_text, fallback_voice)
            await comm.save(output_path)
        asyncio.run(run_edge())

def process_job(doc_ref, job_data):
    job_id = job_data.get('job_id') or doc_ref.id
    title = job_data.get('title', 'EpicSync Video')
    script_text = job_data.get('script', '')
    voice = job_data.get('voice', 'relationship-male')
    aspect_ratio = job_data.get('aspect_ratio', '9:16')
    voice_boost = job_data.get('voice_boost', '100')
    add_captions = job_data.get('add_captions', 'true')
    font_size = job_data.get('font_size', '60')
    pexels_key = job_data.get('pexels_api_key', 'HqD4UjBfH3i9V2lq2jBq0YQp7n3s1k8L5r0a4b9c8d')
    
    print(f'\n🎬 PROCESSING: {job_id} | {title}')
    doc_ref.update({'status': 'RUNNING', 'progress': 15, 'step_text': 'Synthesizing voiceover...'})
    
    work_dir = f'/kaggle/working/job_{job_id}'
    os.makedirs(work_dir, exist_ok=True)
    audio_path = os.path.join(work_dir, 'input.wav')
    
    # 1. Voiceover
    generate_voiceover(script_text, voice, audio_path, doc_ref)
    doc_ref.update({'progress': 40, 'step_text': 'Aligning word timings with Whisper...'})
    
    # 2. Whisper
    import whisper
    wm = whisper.load_model('base')
    res = wm.transcribe(audio_path, word_timestamps=True)
    words_data = [ {'word': w.get('word','').strip(), 'start': w.get('start',0), 'end': w.get('end',0)} for seg in res.get('segments',[]) for w in seg.get('words',[]) ]
    
    # 3. Pexels Clips
    doc_ref.update({'progress': 60, 'step_text': 'Downloading stock B-roll clips...'})
    w_dim, h_dim = (720, 1280) if aspect_ratio == '9:16' else (1280, 720)
    orientation = 'portrait' if aspect_ratio == '9:16' else 'landscape'
    keywords = [w for w in re.findall(r'\b[A-Za-z]{4,}\b', script_text) if w.lower() not in ['this','that','with','from','have','were','will','your','about']][:6] or ['cinematic','nature']
    
    clips_dir = os.path.join(work_dir, 'clips')
    os.makedirs(clips_dir, exist_ok=True)
    downloaded_clips = []
    headers = {'Authorization': pexels_key} if len(pexels_key) > 10 else {}
    
    for idx, kw in enumerate(keywords[:5]):
        try:
            r = requests.get(f'https://api.pexels.com/videos/search?query={kw}&orientation={orientation}&per_page=3', headers=headers, timeout=10)
            if r.ok:
                vids = r.json().get('videos', [])
                if vids:
                    v_files = vids[0].get('video_files', [])
                    best = next((f for f in v_files if f.get('width') == w_dim or f.get('quality') == 'hd'), v_files[0])
                    c_path = os.path.join(clips_dir, f'clip_{idx}.mp4')
                    with requests.get(best.get('link'), stream=True) as vr:
                        with open(c_path, 'wb') as f: shutil.copyfileobj(vr.raw, f)
                    downloaded_clips.append(c_path)
        except: pass
        
    if not downloaded_clips:
        fallback = os.path.join(clips_dir, 'fallback.mp4')
        subprocess.run(f'ffmpeg -y -f lavfi -i color=c=0x111827:s={w_dim}x{h_dim}:d=10 -c:v libx264 -pix_fmt yuv420p {fallback}', shell=True)
        downloaded_clips.append(fallback)
        
    # 4. Render Video
    doc_ref.update({'progress': 80, 'step_text': 'Assembling video and animated subtitles...'})
    import wave
    with wave.open(audio_path, 'r') as f: audio_dur = f.getnframes() / float(f.getframerate())
    
    concat_list = os.path.join(work_dir, 'concat.txt')
    with open(concat_list, 'w') as f:
        for c in downloaded_clips: f.write(f"file '{c}'\n")
        
    raw_video = os.path.join(work_dir, 'raw_video.mp4')
    subprocess.run(f"ffmpeg -y -f concat -safe 0 -stream_loop 10 -i {concat_list} -t {audio_dur} -vf 'scale={w_dim}:{h_dim}:force_original_aspect_ratio=increase,crop={w_dim}:{h_dim}' -c:v libx264 -pix_fmt yuv420p -an {raw_video}", shell=True)
    
    srt_path = os.path.join(work_dir, 'subtitles.srt')
    with open(srt_path, 'w', encoding='utf-8') as f:
        for idx, w in enumerate(words_data):
            st = time.strftime('%H:%M:%S,000', time.gmtime(w['start']))
            et = time.strftime('%H:%M:%S,000', time.gmtime(w['end']))
            f.write(f"{idx+1}\n{st} --> {et}\n{w['word']}\n\n")
            
    final_output = os.path.join(work_dir, f'{job_id}.mp4')
    boost_val = float(voice_boost) / 100.0 if voice_boost else 1.0
    sub_filter = f"-vf \"subtitles={srt_path}:force_style='FontSize={font_size},PrimaryColour=&H00FFFFFF,Alignment=2,MarginV=100'\"" if add_captions == 'true' else ""
    cmd = f"ffmpeg -y -i {raw_video} -i {audio_path} {sub_filter} -filter_complex \"[1:a]volume={boost_val}[aout]\" -map 0:v -map \"[aout]\" -c:v libx264 -preset fast -c:a aac -b:a 192k -shortest {final_output}"
    subprocess.run(cmd, shell=True)
    
    # 5. Upload to HF Dataset
    doc_ref.update({'progress': 95, 'step_text': 'Uploading to Hugging Face Dataset...'})
    hf_path = f'outputs/{job_id}.mp4'
    upload_file(path_or_fileobj=final_output, path_in_repo=hf_path, repo_id=HF_REPO, repo_type='dataset', token=HF_TOKEN)
    
    direct_url = f'https://huggingface.co/datasets/{HF_REPO}/resolve/main/{hf_path}'
    doc_ref.update({
        'status': 'SUCCESS',
        'progress': 100,
        'step_text': 'Video generated successfully!',
        'output_file': direct_url,
        'completedAt': firestore.SERVER_TIMESTAMP
    })
    print(f'🎉 COMPLETED: {job_id}')

# 12-Hour Loop
start_time = time.time()
print('🚀 12-HOUR WORKER ENGINE IS NOW LIVE!')
while time.time() - start_time < 12 * 3600:
    try:
        db.collection('system').document('worker_status').set({
            'status': 'ONLINE',
            'device': 'Kaggle Cloud Worker (12h Session)',
            'last_heartbeat': firestore.SERVER_TIMESTAMP,
            'uptime_minutes': int((time.time() - start_time) / 60)
        }, merge=True)
        
        queued_query = db.collection_group('executions').where('status', '==', 'QUEUED').limit(1).stream()
        found = False
        for d in queued_query:
            found = True
            process_job(d.reference, d.to_dict())
            break
        if not found:
            time.sleep(3)
    except Exception as e:
        print(f'Loop Notice: {e}')
        time.sleep(4)
